# 10年定着予測 - 反復数の最適化とマルチシードOptuna（38_）

**背景**: `37_` で **Train全件学習** が Public を 0.529454 → **0.522659** に改善した
（`data/output/submit_result_report.md` セクション50）。改善の内訳は全件学習 -0.00646 /
シード平均 -0.00034 で、全件学習が本命だったことが確定している。

その `37_` が残した2つの宿題に取り組む。

| # | 課題 | `37_`での状態 |
|---|---|---|
| 1 | **反復数スケールが任意の選択** | `449 × 1.25 = 560` と決め打ち。×1.0や×1.5が良い可能性を未検証 |
| 2 | **Optunaの目的関数が単一シードで不安定** | 検証セットを3.3%変えただけで探索がdepth 4→8、L2が1/26の領域へ飛んだ。シードsd 0.0047〜0.0078に対し探索が拾う差はそれ以下 |

特徴量は `28_`/`37_` と**完全に同一**（441列、ブロックL_v2）。変更はモデルの学習設定のみ。

## 方針

### 課題1: 提出回数を使わずに反復数の感度を測る

全件学習は検証スコアが計算できないため、スケールを総当たりすると提出回数を浪費する。
そこで **80/20 split で反復数を固定した学習曲線**を引き、**最適点の鋭さ**を測る。

- 曲線が平坦なら → スケールの選択はスコアにほとんど影響しない。提出回数を使う価値はない
- 曲線が鋭いなら → 最適点を外していた可能性があり、修正する価値がある

80%学習での最適反復数 `iter_opt` が分かれば、全件学習では件数比 1.25 倍した値を使う。
`37_` の 560 がこの推定値からどれだけ離れているかも分かる。

### 課題2: 目的関数を3シード平均にする

Optunaの各trialで**3シード学習して val logloss の平均**を返す。単一シードの当たり外れで
ハイパーパラメータが選ばれる問題を直接叩く。探索コストは3倍になるが、1 fit あたり数秒なので許容範囲。

得られた新パラメータは、`A_PARAMS`（Public 0.529454/0.522659 を実際に出した既知の良い設定）と
**同じ5シード評価で比較**する。

> ⚠️ **バイアスに注意**: 新パラメータは検証セットに対して探索されているので、その検証セットでの
> 比較は新パラメータ側に有利に出る。検証で勝っても、それだけでは採用理由にならない
> （`23_`〜`26_` で繰り返し痛い目に遭った通り）。最終判断は Public で行う。

## 実行環境

Google Colab Pro の **CPUハイメモリ**ランタイム。想定実行時間は
学習曲線 42回 + Optuna 75回 + 最終学習 10回 ≒ **20〜30分**（特徴量生成を除く）。

> ⚠️ ローカルMacで先行実行しないこと（チェックポイントのDrive同期事故を避けるため）。

In [24]:
!pip install -q catboost optuna

Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 4, in <module>
    from pip._internal.cli.main import main
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 11, in <module>
    from pip._internal.cli.autocompletion import autocomplete
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/autocompletion.py", line 10, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main_parser.py", line 9, in <module>
    from pip._internal.build_env import get_runnable_pip
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/build_env.py", line 14, in <module>
    from pip._vendor.certifi import where
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/certifi/__init__.py", line 1, in <module>
    from .core import contents, where
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/certifi/core.py", line 16, in <module>
 

In [25]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [26]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [28]:
SCRIPT_NAME = "38_iterations_and_multiseed_optuna"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# チェックポイント（日付非依存の固定パス。セッションをまたいで再開できるようにする）
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

RESET_CHECKPOINT = True  # Trueにすると既存チェックポイントを削除して最初から再計算する
if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("チェックポイントを削除しました（全構成を再計算します）")

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")

[2026-08-11 15:07:04] [INFO] === [38_iterations_and_multiseed_optuna] 実験開始 ===


INFO:38_iterations_and_multiseed_optuna:=== [38_iterations_and_multiseed_optuna] 実験開始 ===


[2026-08-11 15:07:04] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260811


INFO:38_iterations_and_multiseed_optuna:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260811


[2026-08-11 15:07:04] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/38_iterations_and_multiseed_optuna_checkpoint.csv


INFO:38_iterations_and_multiseed_optuna:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/38_iterations_and_multiseed_optuna_checkpoint.csv


[2026-08-11 15:07:04] [INFO] チェックポイントは未作成（新規実行）


INFO:38_iterations_and_multiseed_optuna:チェックポイントは未作成（新規実行）


In [29]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-11 15:07:05] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:38_iterations_and_multiseed_optuna:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-11 15:07:05] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:38_iterations_and_multiseed_optuna:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-11 15:07:05] [INFO] 定着率: 0.5647


INFO:38_iterations_and_multiseed_optuna:定着率: 0.5647


[2026-08-11 15:07:05] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:38_iterations_and_multiseed_optuna:Train IDs: 2761, Test IDs: 2502


## 0. 早期退職者の特定（改善3の前提）

`月末在籍状態 == "退職"` の行を持つ社員が、0-23ヶ月の観測期間中に退職した社員。
Train に129名（全員ラベル0）、**Test には0名**。

In [30]:
EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())

logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / {len(train_ids)}名 ({len(EARLY_LEAVER_IDS)/len(train_ids):.1%})")
logger.info(f"Test  早期退職者: {len(_test_early)}名 / {len(test_ids)}名")
_y_idx = train_persona.set_index(ID_COL)[TARGET_COL]
logger.info(f"早期退職者のラベル平均: {_y_idx.loc[list(EARLY_LEAVER_IDS)].mean():.4f}（0.0のはず）")
logger.info(f"定着率: 全体 {y_train.mean():.4f} / 早期退職者を除く {_y_idx[~_y_idx.index.isin(EARLY_LEAVER_IDS)].mean():.4f}")
assert len(_test_early) == 0, "Testに早期退職者が存在する。EDA v6の前提が崩れているので調査すること"

[2026-08-11 15:07:05] [INFO] Train 早期退職者: 129名 / 2761名 (4.7%)


INFO:38_iterations_and_multiseed_optuna:Train 早期退職者: 129名 / 2761名 (4.7%)


[2026-08-11 15:07:05] [INFO] Test  早期退職者: 0名 / 2502名


INFO:38_iterations_and_multiseed_optuna:Test  早期退職者: 0名 / 2502名


[2026-08-11 15:07:05] [INFO] 早期退職者のラベル平均: 0.0000（0.0のはず）


INFO:38_iterations_and_multiseed_optuna:早期退職者のラベル平均: 0.0000（0.0のはず）


[2026-08-11 15:07:05] [INFO] 定着率: 全体 0.5647 / 早期退職者を除く 0.5923


INFO:38_iterations_and_multiseed_optuna:定着率: 全体 0.5647 / 早期退職者を除く 0.5923


## 1. 基本特徴量関数の定義（split非依存、`18_`〜`26_`と同一ロジック）

In [31]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")

✅ split非依存の基本特徴量関数定義完了


In [32]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")

[2026-08-11 15:07:06] [INFO] ------------------------------------------------------------


INFO:38_iterations_and_multiseed_optuna:------------------------------------------------------------


[2026-08-11 15:07:06] [INFO] split非依存の基本特徴量を生成中...


INFO:38_iterations_and_multiseed_optuna:split非依存の基本特徴量を生成中...


[2026-08-11 15:07:06] [INFO] ------------------------------------------------------------


INFO:38_iterations_and_multiseed_optuna:------------------------------------------------------------


[2026-08-11 15:13:10] [INFO] split非依存の基本特徴量生成完了


INFO:38_iterations_and_multiseed_optuna:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`18_`と同一・継続採用）

In [33]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")

[2026-08-11 15:13:10] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:38_iterations_and_multiseed_optuna:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-11 15:13:11] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:38_iterations_and_multiseed_optuna:入社時メモ: SVD累積寄与率=0.760


[2026-08-11 15:13:15] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:38_iterations_and_multiseed_optuna:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-11 15:13:17] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:38_iterations_and_multiseed_optuna:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-11 15:13:17] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:38_iterations_and_multiseed_optuna:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`18_`の勝者を継続採用）

`18_`のステップAで、D_expanded（16指標）がD_original（6指標）・Dなしより2 split平均で最良と判明したため、
以降は常にD_expandedを使う（今回はDブロックの再比較は行わない）。

In [34]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")

[2026-08-11 15:13:17] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:38_iterations_and_multiseed_optuna:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-11 15:15:25] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:38_iterations_and_multiseed_optuna:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（split非依存、`18_`と同一）

In [35]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

[2026-08-11 15:15:25] [INFO] Persona単位の基本特徴量を生成中...


INFO:38_iterations_and_multiseed_optuna:Persona単位の基本特徴量を生成中...


[2026-08-11 15:15:25] [INFO] Persona単位の基本特徴量処理完了


INFO:38_iterations_and_multiseed_optuna:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、v1=`27_`のPublic確認済み版 / v2=抽出拡張版）

`転居許容`フラグの抽出ロジックは`27_`と同一。`希望勤務地`の抽出のみ2種類を用意する：

- **v1**: `27_`・`25_`・`data_exploration_v3/v4/v5`と同一の正規表現（Public 0.529672で確認済み）
- **v2**: v1に加え、「◯◯を希望。」「◯◯勤務を希望。」「◯◯での勤務を希望。」パターンを追加で
  拾う拡張版。未抽出だった152件（train）を目視確認して発見した言い回し。カバー率が
  88.6%→94.1%（train）/ 95.0%（test）に向上し、ダブル悪条件の該当件数も342→354件に増加、
  効果量はp=1.4×10⁻³⁹→1.7×10⁻⁴³・オッズ比0.189→0.178とむしろ強まった。

In [36]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    return m.group(1).strip() if m else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    # reloc_ok_rawはobject dtype(True/False/None混在)のため、~演算子は使わず
    # 明示的な等価比較でTrue/False/欠損を扱う（欠損に対する~はTypeErrorになる）
    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    # 4値カテゴリ（決定木が交互作用を直接学習しやすいよう明示的にエンコード）
    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    # ダブル悪条件フラグ（EDA v5で確認した最も強いシグナル: 転居許容せず AND 勤務地不一致）
    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")
print("L_v1 ダブル悪条件:")
print(train_reloc_v1["転居x勤務地_ダブル悪条件_v1"].value_counts())
print("\nL_v2 ダブル悪条件:")
print(train_reloc_v2["転居x勤務地_ダブル悪条件_v2"].value_counts())

[2026-08-11 15:15:26] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:38_iterations_and_multiseed_optuna:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-11 15:15:26] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:38_iterations_and_multiseed_optuna:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-11 15:15:26] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:38_iterations_and_multiseed_optuna:L_v2: Train (2761, 3), Test (2502, 3)


L_v1 ダブル悪条件:
転居x勤務地_ダブル悪条件_v1
0    2419
1     342
Name: count, dtype: int64

L_v2 ダブル悪条件:
転居x勤務地_ダブル悪条件_v2
0    2407
1     354
Name: count, dtype: int64


## 6. 部署Target Encoding（リーク対策済）と `prepare_split` 関数

`extra_blocks`パラメータで`{"L1"}`/`{"L2"}`を指定し、ベースライン
（D_expanded + TF-IDF A_v1、`18_`の構成、Eなし）に対してL_v1・L_v2のいずれかを単体で追加できるようにする。

In [37]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None, exclude_early_from_val=True):
    '''指定した分割比率で特徴量を組み立てる。

    28_ からの変更点は2つだけ:
      - split_ratio=1.0 を許容（Train全件学習用。ag_tuningは空になる）
      - exclude_early_from_val=True のとき、検証セットから早期退職者を除く（改善3）
    特徴量の作り方そのものは 28_ と完全に同一。
    '''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L1" in extra_blocks:
        tf = tf.merge(train_reloc_v1, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v1, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    # --- 改善3: 検証セットから早期退職者を除く（学習側からは除かない） ---
    if exclude_early_from_val and len(ag_tuning) > 0:
        n_before = len(ag_tuning)
        ag_tuning = ag_tuning[~ag_tuning.index.isin(EARLY_LEAVER_IDS)]
        logger.info(f"  検証セット: {n_before} → {len(ag_tuning)}件（早期退職者{n_before - len(ag_tuning)}名を除外）")

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了（37_版: 全件学習・検証セット補正に対応）")

✅ 部署Target Encoding・prepare_split関数定義完了（37_版: 全件学習・検証セット補正に対応）


## 7. チェックポイント機能（`18_`〜`28_`をベースに、37_で固定スキーマ化）

`28_`までは全configが同じキーを持っていたが、37_ は A / BC / D で記録すべき情報が異なる。
キー構成がバラバラのまま `mode="a"` でCSVに追記すると列がずれて壊れるため、
`RESULT_SCHEMA` に揃えてから書き出す。

In [38]:
RESULT_SCHEMA = ["config", "n_features", "val_score", "val_score_all", "val_score_single",
                   "val_single_mean", "val_single_sd", "best_iter", "n_iterations",
                   "params", "submission_path"]

def make_row(**kwargs):
    """全configで同じ列構成のdictを作る。

    28_ は全configが同じキーを持っていたが、37_ は A / BC / D で必要な情報が異なる。
    キー構成がバラバラのままだと、mode="a" でCSVに追記した際に列がずれて壊れるため、
    固定スキーマに揃えてから書き出す。
    """
    unknown = set(kwargs) - set(RESULT_SCHEMA)
    assert not unknown, f"RESULT_SCHEMAに無いキー: {unknown}"
    row = {k: np.nan for k in RESULT_SCHEMA}
    row.update(kwargs)
    return row


def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=RESULT_SCHEMA)

def save_checkpoint_row(result):
    df = pd.DataFrame([result])[RESULT_SCHEMA]
    write_header = not CHECKPOINT_PATH.exists()
    df.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)

def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label]
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_score={row['val_score']}")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result

print("✅ チェックポイント関数定義完了（37_版: 固定スキーマで列ずれを防止）")

✅ チェックポイント関数定義完了（37_版: 固定スキーマで列ずれを防止）


## 8. モデル関数（37_版）

`28_`の `run_model_config` を3つに分解する。

- `tune_hyperparams`: Optunaで探索（探索空間は`18_`〜`28_`と完全に同一、n_trials=25）
- `fit_holdout`: 80/20で学習し、early stoppingで最良反復数を決める。シードを変えて複数回実行できる
- `fit_full_train`: **Train全件**で学習する（検証セットが無いので反復数は固定、early stoppingなし）

シード平均は、同一パラメータ・同一特徴量のままシードだけ変えたモデルの**予測確率を単純平均**する。
重みを一切学習しないので、`11_`/`12_`/`32_`で失敗した「OOFから重みを学習するアンサンブル」とは
別物であり、過去の教訓には抵触しない。

In [39]:
SEEDS = [42, 2024, 7, 1234, 99]          # 改善2: シード平均に使う5シード
N_TRIALS = 25                            # 18_〜28_と同一
ITER_SCALE_CANDIDATES = {"x125": 1.25, "x100": 1.00}   # 全件学習時の反復数スケール（2761/2208≒1.25）


def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]


def _xy(df, feature_cols):
    return df[feature_cols].fillna(-999), df[TARGET_COL]


def tune_hyperparams(ag_train, ag_val, n_trials=N_TRIALS):
    """Optunaでハイパーパラメータを探索（探索空間は18_〜28_と完全に同一）"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)

    def objective(trial):
        params = {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "iterations": 1000, "random_seed": SEED, "verbose": False,
            "cat_features": obj_cols, "early_stopping_rounds": 50, "task_type": "CPU",
        }
        model = cb.CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        return log_loss(y_va, model.predict_proba(X_va)[:, 1])

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    logger.info(f"  Optuna完了: best_value={study.best_value:.6f}, best_params={study.best_params}")
    return study.best_params


def fit_holdout(ag_train, ag_val, test_features, best_params, seeds):
    """80/20で学習。early stoppingで最良反復数を決め、シードごとの予測を返す"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)
    X_test = test_features[feature_cols].fillna(-999)

    val_preds, test_preds, best_iters = [], [], []
    for seed in seeds:
        model = cb.CatBoostClassifier(
            **best_params, iterations=3000, random_seed=seed, verbose=False,
            cat_features=obj_cols, early_stopping_rounds=100, task_type="CPU",
        )
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        vp = model.predict_proba(X_va)[:, 1]
        val_preds.append(vp)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        best_iters.append(model.get_best_iteration())
        logger.info(f"  seed={seed}: val_logloss={log_loss(y_va, vp):.6f}, best_iteration={best_iters[-1]}")

    return {
        "val_preds": np.array(val_preds), "test_preds": np.array(test_preds),
        "best_iters": best_iters, "y_val": y_va.values, "feature_cols": feature_cols,
    }


def fit_full_train(ag_full, test_features, best_params, n_iterations, seeds):
    """Train全件で学習（改善1）。検証セットが無いので反復数は固定、early stoppingなし"""
    feature_cols = _feature_cols(ag_full)
    obj_cols = [c for c in feature_cols if ag_full[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_full, feature_cols)
    X_test = test_features[feature_cols].fillna(-999)

    test_preds = []
    for seed in seeds:
        model = cb.CatBoostClassifier(
            **best_params, iterations=int(n_iterations), random_seed=seed, verbose=False,
            cat_features=obj_cols, task_type="CPU",
        )
        model.fit(X_tr, y_tr)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        logger.info(f"  seed={seed}: 全件学習完了（iterations={int(n_iterations)}）")
    return np.array(test_preds)


def save_submission(test_index, preds, config_label):
    path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    pd.DataFrame({ID_COL: test_index, TARGET_COL: preds}).to_csv(path, index=False, header=False)
    logger.info(f"  提出ファイル: {path.name}（予測平均={preds.mean():.4f}）")
    return str(path)


print("✅ モデル関数定義完了（tune_hyperparams / fit_holdout / fit_full_train）")

✅ モデル関数定義完了（tune_hyperparams / fit_holdout / fit_full_train）


## 9. 特徴量の組み立て

ブロックは `L2`（= `28_`の `L_v2_extended`、現在の最良）に固定する。

- `split_80_20` × 検証=全体 → config A（`28_`の完全再現）
- `split_80_20` × 検証=生存者のみ → config B / C
- `split_100`（全件） → config D / D2

In [40]:
BLOCK = {"L2"}   # 28_のL_v2_extended（現在の最良）に固定

logger.info("=" * 60)
logger.info("[A用] split_80_20 / 検証=全体（28_と同一）")
ag_train_80, ag_val_all, test_features = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=False)

logger.info("[B,C用] split_80_20 / 検証=生存者のみ")
ag_train_80b, ag_val_surv, _ = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("[D用] 全件学習（検証セットなし）")
ag_full, ag_empty, test_features_full = prepare_split(1.0, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("-" * 60)
logger.info(f"A: train={len(ag_train_80)}, val={len(ag_val_all)}（早期退職者を含む）")
logger.info(f"B/C: train={len(ag_train_80b)}, val={len(ag_val_surv)}（生存者のみ）")
logger.info(f"D: train={len(ag_full)}（全件）, val={len(ag_empty)}（空）")
logger.info(f"特徴量数: {len(_feature_cols(ag_train_80))}")

# 生存者マスク（Aの検証予測を生存者だけで採点し直すのに使う）
SURV_MASK_A = ~ag_val_all.index.isin(EARLY_LEAVER_IDS)
assert len(ag_train_80) == len(ag_train_80b), "A と B/C の学習データは同一のはず"
assert len(ag_empty) == 0, "全件学習のときは検証セットが空のはず"

[2026-08-11 15:15:27] [INFO] ============================================================


INFO:38_iterations_and_multiseed_optuna:============================================================


[2026-08-11 15:15:27] [INFO] [A用] split_80_20 / 検証=全体（28_と同一）


INFO:38_iterations_and_multiseed_optuna:[A用] split_80_20 / 検証=全体（28_と同一）


[2026-08-11 15:15:27] [INFO] [B,C用] split_80_20 / 検証=生存者のみ


...

INFO:38_iterations_and_multiseed_optuna:B/C: train=2208, val=535（生存者のみ）


[2026-08-11 15:15:27] [INFO] D: train=2761（全件）, val=0（空）


INFO:38_iterations_and_multiseed_optuna:D: train=2761（全件）, val=0（空）


[2026-08-11 15:15:27] [INFO] 特徴量数: 441


INFO:38_iterations_and_multiseed_optuna:特徴量数: 441


## 8b. `37_`から引き継ぐ既知の良いハイパーパラメータ

`A_PARAMS` は `37_` の config A（= `28_` の完全再現、Public 0.529454）で Optuna が選んだ設定で、
そのまま全件学習に載せた `D3` が現在の最良（Public 0.522659）を出している。本ノートブックの
基準点として使う。

In [41]:
# 37_ の config A で選ばれた設定（= 28_ と同一。Public 0.529454 / D3で 0.522659 の実績）
A_PARAMS = {
    "depth": 4,
    "learning_rate": 0.03518359458951149,
    "l2_leaf_reg": 2.217690447016724,
    "border_count": 218,
    "bagging_temperature": 0.6787467566574921,
    "random_strength": 1.438494697238285,
}
D3_ITERATIONS = 560          # 37_ D3 が使った反復数（= best_iter平均449 × 1.25）
FULL_TRAIN_SCALE = 2761 / 2208   # 全件学習時の件数比（≒1.2505）

logger.info(f"A_PARAMS = {A_PARAMS}")
logger.info(f"37_ D3 の反復数 = {D3_ITERATIONS}, 全件スケール = {FULL_TRAIN_SCALE:.4f}")

def fit_fixed_iters(ag_train, ag_val, params, n_iter, seeds):
    """反復数を固定して学習し（early stoppingなし）、val logloss を返す"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)
    scores, preds = [], []
    for seed in seeds:
        m = cb.CatBoostClassifier(**params, iterations=int(n_iter), random_seed=seed,
                                  verbose=False, cat_features=obj_cols, task_type="CPU")
        m.fit(X_tr, y_tr)
        p = m.predict_proba(X_va)[:, 1]
        preds.append(p); scores.append(log_loss(y_va, p))
    avg = log_loss(y_va, np.mean(preds, axis=0))
    return float(np.mean(scores)), float(np.std(scores)), avg

print("✅ fit_fixed_iters 定義完了")

[2026-08-11 15:15:27] [INFO] A_PARAMS = {'depth': 4, 'learning_rate': 0.03518359458951149, 'l2_leaf_reg': 2.217690447016724, 'border_count': 218, 'bagging_temperature': 0.6787467566574921, 'random_strength': 1.438494697238285}


INFO:38_iterations_and_multiseed_optuna:A_PARAMS = {'depth': 4, 'learning_rate': 0.03518359458951149, 'l2_leaf_reg': 2.217690447016724, 'border_count': 218, 'bagging_temperature': 0.6787467566574921, 'random_strength': 1.438494697238285}


[2026-08-11 15:15:27] [INFO] 37_ D3 の反復数 = 560, 全件スケール = 1.2505


INFO:38_iterations_and_multiseed_optuna:37_ D3 の反復数 = 560, 全件スケール = 1.2505


✅ fit_fixed_iters 定義完了


## 15. 課題1: 反復数の感度曲線（提出回数を消費しない）

`A_PARAMS` を固定し、80/20 split・early stopping なしで反復数だけを変えて val logloss を測る。
検証は生存者のみ535名、各点3シード。

見るべきは**最小値の位置**そのものより、**最小値付近がどれだけ平坦か**。
平坦なら `37_` の 560 という決め打ちは問題なかったことになり、この方向は打ち切ってよい。

In [42]:
CURVE_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_iter_curve.csv"  # 日付非依存（チェックポイントと同じ寿命）

ITER_GRID = [150, 250, 350, 450, 560, 700, 900, 1200]
CURVE_SEEDS = [42, 2024, 7]

def run_iter_curve():
    logger.info("=" * 60); logger.info("[課題1] 反復数の感度曲線（A_PARAMS, 80/20, 生存者のみ検証）")
    rows = []
    for n_iter in ITER_GRID:
        mean_s, sd_s, avg_s = fit_fixed_iters(ag_train_80b, ag_val_surv, A_PARAMS, n_iter, CURVE_SEEDS)
        rows.append({"iterations": n_iter, "val_単一シード平均": mean_s, "val_sd": sd_s, "val_シード平均": avg_s})
        logger.info(f"  iterations={n_iter:5d}: 単一平均={mean_s:.6f} (sd={sd_s:.6f}), シード平均={avg_s:.6f}")
    df = pd.DataFrame(rows)
    df.to_csv(CURVE_PATH, index=False)
    best = df.loc[df["val_シード平均"].idxmin()]
    return make_row(config="iter_curve", n_features=len(_feature_cols(ag_train_80b)),
                    val_score=float(best["val_シード平均"]), best_iter=float(best["iterations"]),
                    params=json.dumps(A_PARAMS), submission_path=str(CURVE_PATH))

result_curve = run_or_resume("iter_curve", run_iter_curve)
curve_df = pd.read_csv(CURVE_PATH)
display(curve_df.round(6))

ITER_OPT_80 = int(result_curve["best_iter"])
best_score = curve_df["val_シード平均"].min()
worst_near = curve_df[(curve_df.iterations >= 350) & (curve_df.iterations <= 900)]["val_シード平均"].max()
print(f"\n80%学習での最適反復数: {ITER_OPT_80}")
print(f"350〜900の範囲での振れ幅: {worst_near - best_score:.6f}")
print(f"  → シードsd({curve_df['val_sd'].mean():.6f})より小さければ、反復数の選択はほぼ影響しない")
print(f"\n全件学習での推奨反復数: {int(ITER_OPT_80 * FULL_TRAIN_SCALE)}  （37_ D3 は {D3_ITERATIONS}）")

[2026-08-11 15:15:28] [INFO] ============================================================


INFO:38_iterations_and_multiseed_optuna:============================================================


[2026-08-11 15:15:28] [INFO] [課題1] 反復数の感度曲線（A_PARAMS, 80/20, 生存者のみ検証）


INFO:38_iterations_and_multiseed_optuna:[課題1] 反復数の感度曲線（A_PARAMS, 80/20, 生存者のみ検証）


[2026-08-11 15:15:32] [INFO]   iterations=  150: 単一平均=0.553898 (sd=0.001881), シード平均=0.553105


...

INFO:38_iterations_and_multiseed_optuna:  iterations=  900: 単一平均=0.524649 (sd=0.004671), シード平均=0.521024


[2026-08-11 15:17:30] [INFO]   iterations= 1200: 単一平均=0.533364 (sd=0.005692), シード平均=0.528904


INFO:38_iterations_and_multiseed_optuna:  iterations= 1200: 単一平均=0.533364 (sd=0.005692), シード平均=0.528904


,iterations,val_単一シード平均,val_sd,val_シード平均
0,150,0.553898,0.001881,0.553105
1,250,0.528301,0.004916,0.527185
2,350,0.521851,0.005227,0.520232
3,450,0.520821,0.004804,0.518820
4,560,0.520012,0.003790,0.517498
5,700,0.521424,0.004407,0.518223
6,900,0.524649,0.004671,0.521024
7,1200,0.533364,0.005692,0.528904



80%学習での最適反復数: 560
350〜900の範囲での振れ幅: 0.003525
  → シードsd(0.004423)より小さければ、反復数の選択はほぼ影響しない

全件学習での推奨反復数: 700  （37_ D3 は 560）


## 16. 課題2: マルチシードOptuna

各trialで3シード学習し、**val logloss の平均**を目的関数にする。探索空間は `18_`〜`37_` と完全に同一。

In [43]:
TUNE_SEEDS = [42, 2024, 7]

def tune_hyperparams_multiseed(ag_train, ag_val, n_trials=N_TRIALS, tune_seeds=TUNE_SEEDS):
    """各trialで複数シード学習し、val loglossの平均を返す（37_の単一シード版を安定化）"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)

    def objective(trial):
        base = {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        }
        scores = []
        for seed in tune_seeds:
            m = cb.CatBoostClassifier(**base, iterations=1000, random_seed=seed, verbose=False,
                                      cat_features=obj_cols, early_stopping_rounds=50, task_type="CPU")
            m.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
            scores.append(log_loss(y_va, m.predict_proba(X_va)[:, 1]))
        return float(np.mean(scores))

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    logger.info(f"  マルチシードOptuna完了: best_value={study.best_value:.6f}")
    logger.info(f"  best_params={study.best_params}")
    return study.best_params


def run_multiseed_optuna():
    logger.info("=" * 60); logger.info(f"[課題2] マルチシードOptuna（{len(TUNE_SEEDS)}シード平均を目的関数, {N_TRIALS}試行）")
    params_new = tune_hyperparams_multiseed(ag_train_80b, ag_val_surv)

    # 新パラメータとA_PARAMSを、同じ5シード評価で比較する
    out_new = fit_holdout(ag_train_80b, ag_val_surv, test_features, params_new, seeds=SEEDS)
    new_single = [log_loss(out_new["y_val"], vp) for vp in out_new["val_preds"]]
    new_avg = log_loss(out_new["y_val"], out_new["val_preds"].mean(axis=0))
    logger.info(f"  [新パラメータ] 単一シード平均={np.mean(new_single):.6f} (sd={np.std(new_single):.6f}), "
                f"シード平均={new_avg:.6f}, best_iter平均={np.mean(out_new['best_iters']):.0f}")

    path = save_submission(test_features.index, out_new["test_preds"].mean(axis=0), "G_newparams_seedavg_80pct")
    np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_G_newparams_seedavg_80pct_valpreds.npy",
            out_new["val_preds"].mean(axis=0))

    return make_row(config="G_multiseed_optuna", n_features=len(out_new["feature_cols"]),
                    val_score=new_avg, val_score_single=new_single[0],
                    val_single_mean=float(np.mean(new_single)), val_single_sd=float(np.std(new_single)),
                    best_iter=float(np.mean(out_new["best_iters"])), params=json.dumps(params_new),
                    submission_path=path)

result_G = run_or_resume("G_multiseed_optuna", run_multiseed_optuna)
print("新パラメータ:", json.loads(result_G["params"]))
print("A_PARAMS   :", A_PARAMS)

[2026-08-11 15:17:30] [INFO] ============================================================


INFO:38_iterations_and_multiseed_optuna:============================================================


[2026-08-11 15:17:30] [INFO] [課題2] マルチシードOptuna（3シード平均を目的関数, 25試行）


INFO:38_iterations_and_multiseed_optuna:[課題2] マルチシードOptuna（3シード平均を目的関数, 25試行）


[2026-08-11 15:24:23] [INFO]   マルチシードOptuna完了: best_value=0.516072


...

[2026-08-11 15:24:40] [INFO]   [新パラメータ] 単一シード平均=0.518878 (sd=0.004193), シード平均=0.515703, best_iter平均=336


INFO:38_iterations_and_multiseed_optuna:  [新パラメータ] 単一シード平均=0.518878 (sd=0.004193), シード平均=0.515703, best_iter平均=336


[2026-08-11 15:24:40] [INFO]   提出ファイル: 20260811_38_iterations_and_multiseed_optuna_G_newparams_seedavg_80pct.csv（予測平均=0.5957）


INFO:38_iterations_and_multiseed_optuna:  提出ファイル: 20260811_38_iterations_and_multiseed_optuna_G_newparams_seedavg_80pct.csv（予測平均=0.5957）


新パラメータ: {'depth': 4, 'learning_rate': 0.04169457237610343, 'l2_leaf_reg': 2.184065334113083, 'border_count': 161, 'bagging_temperature': 0.7140904730532207, 'random_strength': 1.1821976868990642}
A_PARAMS   : {'depth': 4, 'learning_rate': 0.03518359458951149, 'l2_leaf_reg': 2.217690447016724, 'border_count': 218, 'bagging_temperature': 0.6787467566574921, 'random_strength': 1.438494697238285}


### A_PARAMS との公平な比較

新パラメータは検証セットに対して探索されている（＝その検証セットでは有利に出る）ので、
`A_PARAMS` も**同じ5シード評価**にかけて並べる。`37_` の C2 が既にこれを実施済み
（val シード平均 0.517698）なので、その値と比べる。

In [44]:
def run_aparams_5seed_ref():
    logger.info("=" * 60); logger.info("[参照] A_PARAMS の5シード評価（37_ C2 の再現）")
    out = fit_holdout(ag_train_80b, ag_val_surv, test_features, A_PARAMS, seeds=SEEDS)
    single = [log_loss(out["y_val"], vp) for vp in out["val_preds"]]
    avg = log_loss(out["y_val"], out["val_preds"].mean(axis=0))
    logger.info(f"  [A_PARAMS] 単一シード平均={np.mean(single):.6f} (sd={np.std(single):.6f}), シード平均={avg:.6f}")
    return make_row(config="ref_Aparams_5seed", n_features=len(out["feature_cols"]),
                    val_score=avg, val_single_mean=float(np.mean(single)),
                    val_single_sd=float(np.std(single)),
                    best_iter=float(np.mean(out["best_iters"])), params=json.dumps(A_PARAMS),
                    submission_path="(提出なし・参照用)")

result_ref = run_or_resume("ref_Aparams_5seed", run_aparams_5seed_ref)

cmp_df = pd.DataFrame([
    {"params": "A_PARAMS（現最良の設定）", "val_単一シード平均": float(result_ref["val_single_mean"]),
     "val_sd": float(result_ref["val_single_sd"]), "val_シード平均": float(result_ref["val_score"]),
     "best_iter平均": float(result_ref["best_iter"])},
    {"params": "新パラメータ（マルチシードOptuna）", "val_単一シード平均": float(result_G["val_single_mean"]),
     "val_sd": float(result_G["val_single_sd"]), "val_シード平均": float(result_G["val_score"]),
     "best_iter平均": float(result_G["best_iter"])},
])
display(cmp_df.round(6))
print(f"\n37_ C2 の実測値: 0.517698（A_PARAMS 5シード平均）← 上表のA_PARAMSと一致すれば再現性OK")
print(f"新パラメータの改善幅: {float(result_G['val_score']) - float(result_ref['val_score']):+.6f}")
print("  ※ 新パラメータは この検証セットで探索されているため有利に出る。差がシードsd程度なら採用しない")

[2026-08-11 15:24:40] [INFO] ============================================================


INFO:38_iterations_and_multiseed_optuna:============================================================


[2026-08-11 15:24:40] [INFO] [参照] A_PARAMS の5シード評価（37_ C2 の再現）


INFO:38_iterations_and_multiseed_optuna:[参照] A_PARAMS の5シード評価（37_ C2 の再現）


[2026-08-11 15:24:45] [INFO]   seed=42: val_logloss=0.514254, best_iteration=584


...

INFO:38_iterations_and_multiseed_optuna:  seed=99: val_logloss=0.525715, best_iteration=366


[2026-08-11 15:25:03] [INFO]   [A_PARAMS] 単一シード平均=0.520469 (sd=0.004670), シード平均=0.517698


INFO:38_iterations_and_multiseed_optuna:  [A_PARAMS] 単一シード平均=0.520469 (sd=0.004670), シード平均=0.517698


,params,val_単一シード平均,val_sd,val_シード平均,best_iter平均
0,A_PARAMS（現最良の設定）,0.520469,0.004670,0.517698,448.6
1,新パラメータ（マルチシードOptuna）,0.518878,0.004193,0.515703,335.8



37_ C2 の実測値: 0.517698（A_PARAMS 5シード平均）← 上表のA_PARAMSと一致すれば再現性OK
新パラメータの改善幅: -0.001995
  ※ 新パラメータは この検証セットで探索されているため有利に出る。差がシードsd程度なら採用しない


## 17. 最終提出候補（全件学習）

課題1で得た最適反復数と、課題2で得た新パラメータを、それぞれ全件学習に載せる。

In [45]:
def make_full_config(tag, params, n_iter):
    def _run():
        logger.info("=" * 60)
        logger.info(f"[{tag}] Train全件学習: iterations={n_iter}, params={params}")
        preds = fit_full_train(ag_full, test_features_full, params, n_iter, seeds=SEEDS)
        path = save_submission(test_features_full.index, preds.mean(axis=0), tag)
        return make_row(config=tag, n_features=len(_feature_cols(ag_full)),
                        n_iterations=n_iter, params=json.dumps(params), submission_path=path)
    return _run

ITER_FULL_A = int(ITER_OPT_80 * FULL_TRAIN_SCALE)
ITER_FULL_NEW = int(float(result_G["best_iter"]) * FULL_TRAIN_SCALE)
params_new = json.loads(result_G["params"])

result_H1 = run_or_resume(f"H1_Aparams_full_iter{ITER_FULL_A}",
                          make_full_config(f"H1_Aparams_full_iter{ITER_FULL_A}", A_PARAMS, ITER_FULL_A))
result_H2 = run_or_resume(f"H2_newparams_full_iter{ITER_FULL_NEW}",
                          make_full_config(f"H2_newparams_full_iter{ITER_FULL_NEW}", params_new, ITER_FULL_NEW))

print(f"H1 (A設定・反復数最適化):  iterations={ITER_FULL_A}  （37_ D3 は {D3_ITERATIONS}）")
print(f"H2 (新設定・全件学習)   :  iterations={ITER_FULL_NEW}")

[2026-08-11 15:25:03] [INFO] ============================================================


INFO:38_iterations_and_multiseed_optuna:============================================================


[2026-08-11 15:25:03] [INFO] [H1_Aparams_full_iter700] Train全件学習: iterations=700, params={'depth': 4, 'learning_rate': 0.03518359458951149, 'l2_leaf_reg': 2.217690447016724, 'border_count': 218, 'bagging_temperature': 0.6787467566574921, 'random_strength': 1.438494697238285}


INFO:38_iterations_and_multiseed_optuna:[H1_Aparams_full_iter700] Train全件学習: iterations=700, params={'depth': 4, 'learning_rate': 0.03518359458951149, 'l2_leaf_reg': 2.217690447016724, 'border_count': 218, 'bagging_temperature': 0.6787467566574921, 'random_strength': 1.438494697238285}


[2026-08-11 15:25:09] [INFO]   seed=42: 全件学習完了（iterations=700）


...

[2026-08-11 15:25:50] [INFO]   seed=99: 全件学習完了（iterations=419）


INFO:38_iterations_and_multiseed_optuna:  seed=99: 全件学習完了（iterations=419）


[2026-08-11 15:25:50] [INFO]   提出ファイル: 20260811_38_iterations_and_multiseed_optuna_H2_newparams_full_iter419.csv（予測平均=0.5883）


INFO:38_iterations_and_multiseed_optuna:  提出ファイル: 20260811_38_iterations_and_multiseed_optuna_H2_newparams_full_iter419.csv（予測平均=0.5883）


H1 (A設定・反復数最適化):  iterations=700  （37_ D3 は 560）
H2 (新設定・全件学習)   :  iterations=419


## 18. 総合結果・提出方針

In [46]:
summary = pd.DataFrame([
    {"config": "37_ D3（現在の最良）", "params": "A_PARAMS", "学習": "全件", "iterations": D3_ITERATIONS,
     "Public": 0.522659, "submission": "（提出済み）"},
    {"config": "H1_Aparams_full", "params": "A_PARAMS", "学習": "全件", "iterations": ITER_FULL_A,
     "Public": np.nan, "submission": Path(result_H1["submission_path"]).name},
    {"config": "H2_newparams_full", "params": "新（マルチシード）", "学習": "全件", "iterations": ITER_FULL_NEW,
     "Public": np.nan, "submission": Path(result_H2["submission_path"]).name},
    {"config": "G_newparams_seedavg_80pct", "params": "新（マルチシード）", "学習": "先頭80%", "iterations": "early stopping",
     "Public": np.nan, "submission": Path(result_G["submission_path"]).name},
])
display(summary)
summary.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_summary.csv", index=False)

print("\n【提出の判断基準】")
print(f"1. 反復数: 350〜900での振れ幅が シードsd({curve_df['val_sd'].mean():.6f}) 以下なら、")
print(f"   H1 は 37_ D3(iterations={D3_ITERATIONS}) とほぼ同じ結果になるはず。提出する価値は低い。")
print(f"   H1のiterations={ITER_FULL_A} が {D3_ITERATIONS} と大きく違う場合のみ提出する。")
print(f"2. 新パラメータ: val改善幅 {float(result_G['val_score']) - float(result_ref['val_score']):+.6f} が")
print(f"   シードsd({float(result_ref['val_single_sd']):.6f}) を明確に超える場合のみ H2 を提出する。")
print("   超えない場合、探索バイアス込みでこの程度の差しか出ていない＝実質改善なしと判断する。")

,config,params,学習,iterations,Public,submission
0,37_ D3（現在の最良）,A_PARAMS,全件,560,0.522659,（提出済み）
1,H1_Aparams_full,A_PARAMS,全件,700,NaN,20260811_38_iterations_and_multiseed_optuna_H1...
2,H2_newparams_full,新（マルチシード）,全件,419,NaN,20260811_38_iterations_and_multiseed_optuna_H2...
3,G_newparams_seedavg_80pct,新（マルチシード）,先頭80%,early stopping,NaN,20260811_38_iterations_and_multiseed_optuna_G_...



【提出の判断基準】
1. 反復数: 350〜900での振れ幅が シードsd(0.004423) 以下なら、
   H1 は 37_ D3(iterations=560) とほぼ同じ結果になるはず。提出する価値は低い。
   H1のiterations=700 が 560 と大きく違う場合のみ提出する。
2. 新パラメータ: val改善幅 -0.001995 が
   シードsd(0.004670) を明確に超える場合のみ H2 を提出する。
   超えない場合、探索バイアス込みでこの程度の差しか出ていない＝実質改善なしと判断する。


## 19. 提出方針と注意

### 提出の判断は「検証で勝ったか」ではなく「シード分散を超えたか」

`37_` で確定した通り、この特徴量セットでのシードsdは **0.0047〜0.0078**。
これは `20_`〜`35_` で採否を判定してきた効果量より大きい。したがって:

- **反復数（課題1）**: 感度曲線の 350〜900 の範囲での振れ幅がシードsd以下なら、
  `37_` の 560 という決め打ちは問題なかったということ。**この方向は打ち切り**、提出しない。
- **新パラメータ（課題2）**: 新パラメータは検証セットに対して探索されているので、
  その検証セットで多少良く出るのは当たり前。**シードsdを明確に超える改善**が出た場合のみ H2 を提出する。

### 予想される結末

正直なところ、**どちらも「効果なし」に終わる可能性が相応にある**。

- 課題1は、GBDTの学習曲線が最適点付近で平坦なのが普通なので、560 が多少ズレていても影響は小さいと予想される。
- 課題2は、探索の安定化であって探索空間の拡張ではないため、`A_PARAMS` より明確に良い設定が
  見つかる保証はない。むしろ `A_PARAMS` は既に Public で2回（0.529454 / 0.522659）実績のある設定である。

その場合の結論は「**`37_` D3 が現時点の到達点で、モデリング手続き側の改善は出尽くした**」となり、
それはそれで探索を打ち切る根拠として価値がある。

### 次のアクション

- H1/H2 を提出した場合は `data/output/submit_result_report.md` に追記する。
- 両方とも効果がなかった場合、残る選択肢は
  (a) `16_`〜`35_` で却下した特徴量ブロックを**全件学習＋シード平均の新プロトコル**で再検証する
  （コストは高いが、却下判定が 80%学習・単一シードで行われていたため結論が変わる可能性がある）、
  (b) 特徴量探索に戻る（EDA v6 でテキスト・交絡候補・行動系はほぼ枯渇を確認済み）
  のいずれかになる。